In [49]:
import pandas as pd
df =pd.read_excel("./1.xlsx",encoding='gbk')

In [50]:
#keywords=['26283121778226','26076662786226','26067223087226','25779902720226','26066979183226','25785921087226','26066692527226','26067006475226']
 #10000229167469,10000229407532,10000265046381,10000270409428,10000269553640,10000273809516
#df['商品ID'].isin(keywords)
#df['商品ID'] = df['商品ID'].astype(str)


#删除其他行
#df.drop(index= df[ df['商品ID'].isin(keywords)==False].index,inplace=True)
#df.reset_index(drop=True,inplace= True)

In [51]:
df["实付款"] = df["实付款"].astype(str).str.replace("¥", "").astype(float)

In [52]:
from datetime import datetime
df["订单创建时间"] = pd.to_datetime(df["订单创建时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单创建时间"] = df["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))
df1 = pd.read_excel("商品编码.xlsx")
df1.rename(columns={"商品名":"产品名称","商家编码":"SKU编码"},inplace=True)
df1 = df1[["SKU编码","产品名称"]]
df= pd.merge(df,df1,on="SKU编码",how="left")

In [53]:
# 缺码表
lackCode = df[df['产品名称'].isnull()]['SKU编码'].value_counts()
lackCode

Series([], Name: SKU编码, dtype: int64)

In [54]:
lackCode.to_excel('缺码表.xlsx')

In [55]:
df['订单数'] = df['成交数量']

In [56]:
df["下单单量"] = df['订单数']

In [57]:
# 清除未支付订单
from datetime import datetime
df["订单支付时间"] = df["订单支付时间"].fillna(pd.to_datetime('1970-01-01'))
df["订单支付时间"] = pd.to_datetime(df["订单支付时间"],format = "%Y-%m-%d %H:%M:%S")
df["订单支付时间"] = df["订单支付时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [58]:
df["发货时间"] = df["发货时间"].fillna(pd.to_datetime('1970-01-01'))
df["发货时间"] = pd.to_datetime(df["发货时间"],format = "%Y-%m-%d %H:%M:%S")
df["发货时间"] = df["发货时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [59]:
df["支付单量"] = 0
df["发货单量"] = 0
df["支付金额"] = 0
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01":
        df.loc[i,"发货单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01":
        df.loc[i,"支付单量"] = df.loc[i,'订单数']
        df.loc[i,"支付金额"] = df.loc[i,'实付款']

In [60]:
df["售后状态"] = df["售后状态"].astype(str)

In [61]:
df["商品ID"] = df["商品ID"].astype(str)

In [62]:
df["售后状态"].value_counts()

nan      1399
退款成功      543
待卖家收货      51
待买家退货      19
退款关闭        8
待买家收货       3
待买家处理       2
待卖家处理       1
Name: 售后状态, dtype: int64

In [63]:
df["商品ID"].value_counts()

27128932157553    1337
27039882950553     313
27064084342553     233
27057988636553     119
27160209961553       9
27047154810553       8
27014358443553       7
Name: 商品ID, dtype: int64

In [64]:
df['物流信息'].value_counts()

已签收      916
运输中      364
未知       260
已发货      128
商家已接单    124
待取件      100
派送中       77
已退签       37
退回中       18
物流异常       1
商家拣货中      1
Name: 物流信息, dtype: int64

In [65]:
for i in range (0,len(df)):
    if df.loc[i,"订单状态"] == "已收货" and (df.loc[i,'售后状态'] == 'nan' or df.loc[i,'售后状态'] == '退款关闭'):
        df.loc[i,"订单状态"] = "交易成功"

In [66]:
df["已签收"] = 0
df["发货中"] = 0
df["售后中"] = 0
df["已退款"] = 0
df["预计签收数量"] = 0
df["发货前退款单量"] = 0
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "交易成功" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"已签收"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"订单状态"] == "已发货" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"发货中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"售后状态"] == "待买家处理" or df.loc[i,"售后状态"] == "待买家退货" or df.loc[i,"售后状态"] == "待卖家处理" or df.loc[i,"售后状态"]=="待卖家收货" or df.loc[i,"售后状态"]=="待平台判定":
        df.loc[i,"售后中"] = df.loc[i,"订单数"]
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"已退款"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"订单支付时间"] != "1970-01-01"  and  df.loc[i,"发货时间"] == "1970-01-01" and df.loc[i,"售后状态"] == "退款成功":
        df.loc[i,"发货前退款单量"] = df.loc[i,'订单数']
for i in range(0,len(df)):
    if df.loc[i,"发货时间"] != "1970-01-01" and (df.loc[i,"售后状态"]== "nan" or df.loc[i,"售后状态"]=="退款关闭"):
        df.loc[i,"预计签收单量"] = df.loc[i,"订单数"]

In [67]:
import numpy as np
result = pd.pivot_table(df,index=['订单创建时间',"产品名称"] , values = ['下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额'] , aggfunc = np.sum)
result = result.reset_index()
result = result[['订单创建时间',"产品名称",'下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]

In [68]:
result.to_excel("111.xlsx",index=False)

In [69]:
# 拼接店铺消耗表
df_consume = pd.read_excel("快手店铺消耗26号.xlsx")
df_consume = df_consume[['日期','产品','花费','转化数']]
temp = df_consume.日期.str.extract('(\d+).*?(\d+).*?(\d+)') 
df_consume['订单创建时间'] = temp[0] + "-" + temp[1] + '-' + temp[2]
df_consume.rename(columns={"产品":"产品名称"},inplace=True)
del df_consume['日期']
df_consume = pd.pivot_table(df_consume,index=['订单创建时间','产品名称'] , values = ['花费','转化数'] , aggfunc = np.sum)
df_consume = df_consume.reset_index()
df_consume["订单创建时间"] = pd.to_datetime(df_consume["订单创建时间"],format = "%Y-%m-%d")
df_consume["订单创建时间"] = df_consume["订单创建时间"].map(lambda x: x.strftime('%Y-%m-%d'))

In [70]:
final = pd.merge(result,df_consume,on = ['订单创建时间','产品名称'],how = 'left')
final = final[['订单创建时间',"产品名称",'转化数','下单单量','支付单量','发货单量','已签收','发货中','售后中','已退款','发货前退款单量',"预计签收单量",'支付金额']]
final.rename(columns={"转化数":"总广告单量"},inplace=True)

In [71]:
final

,订单创建时间,产品名称,总广告单量,下单单量,支付单量,发货单量,已签收,发货中,售后中,已退款,发货前退款单量,预计签收单量,支付金额
0,2026-10-29,915保暖打底衫,16.0,18,16,13,9,0,0,4,3,9.0,1288.4
1,2026-10-30,915保暖打底衫,16.0,17,16,15,11,0,0,4,1,11.0,1278.4
2,2026-10-31,915保暖打底衫,10.0,11,10,10,5,0,0,5,0,5.0,799.0
3,2026-11-01,915保暖打底衫,19.0,23,19,19,15,1,0,3,0,16.0,1518.1
4,2026-11-02,915保暖打底衫,25.0,31,26,25,18,0,0,7,1,18.0,2077.4
5,2026-11-03,915保暖打底衫,0.0,14,13,13,10,0,0,3,0,10.0,1038.7
6,2026-11-04,915保暖打底衫,142.0,154,142,137,72,1,1,62,5,73.0,11395.8
7,2026-11-05,915保暖打底衫,131.0,150,132,123,63,1,1,58,9,64.0,10566.8
8,2026-11-06,915保暖打底衫,67.0,77,68,61,36,1,2,22,7,37.0,5443.2
9,2026-11-07,915保暖打底衫,133.0,148,137,125,61,2,1,61,12,63.0,10986.3


In [72]:
final.to_excel('直播签收汇总.xlsx')